# FICOS Fresh Pipeline System Test

This notebook is a **new experiment**. It reads only `data/modeling_dataset.csv` and computes fresh walk-forward forecasts, uncertainty, voyage opportunities, MILP/CVaR decisions, and OOS timing economics.

It does **not** read prior predictions, optimization outputs, economic reports, or previous experiment CSVs.

The planning fields that do not exist in the canonical data are explicitly labeled `SCENARIO_ASSUMPTION`.

In [ ]:
import os, subprocess
from pathlib import Path
clone_root = Path('/content/FICOS-Platform')
if not (Path.cwd() / 'data' / 'modeling_dataset.csv').exists():
    if not clone_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/SSOHEB/FICOS-Platform.git', str(clone_root)], check=True)
    os.chdir(clone_root)
print('Repository root:', Path.cwd())

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'modeling_dataset.csv').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / 'data' / 'modeling_dataset.csv'
OUT = ROOT / 'outputs' / 'experiments' / 'fresh_pipeline_notebook'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
EXPECTED_DATASET_SHA = 'e0f4c91eed7b4919200472c3fe7e0735e4fd12433383727b58f73c2fd8945fd5'

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

assert sha256(DATA_PATH) == EXPECTED_DATASET_SHA, 'Canonical dataset identity changed'
print('Fresh experiment output:', OUT)
print('Dataset SHA-256:', sha256(DATA_PATH))

In [ ]:
# Load the canonical data directly. No prior output files are read.
df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
feature_cols = [c for c in df.columns if c != 'date' and not c.startswith('target_') and not c.startswith('dir_')]
assert len(feature_cols) == 441
VESSELS = ['panamax', 'supramax', 'handy', 'cape']
HORIZON = 1
OOS_YEARS = [2021, 2022, 2023, 2024, 2025]
print('Rows:', len(df), 'Features:', len(feature_cols))
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
print('OOS years:', OOS_YEARS)

## 1. Forecast: strict expanding walk-forward

For each vessel and test year, preprocessing and RF fitting happen inside that fold. The calibration tail is taken only from dates before the test year. The test year is never used for fitting, threshold selection, or uncertainty calibration.

In [ ]:
def fit_fold_predict(train_df, test_df, vessel):
    target = f'target_{vessel}_{HORIZON}d'
    train_df = train_df[train_df[target].notna()].copy()
    test_df = test_df[test_df[target].notna()].copy()
    calibration_n = min(200, max(30, len(train_df) // 5))
    core = train_df.iloc[:-calibration_n]
    calibration = train_df.iloc[-calibration_n:]
    X_core = core[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_cal = calibration[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaler = StandardScaler().fit(X_core)
    selector = SelectKBest(f_regression, k=30).fit(scaler.transform(X_core), core[target])
    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=1)
    model.fit(selector.transform(scaler.transform(X_core)), core[target])
    cal_pred = model.predict(selector.transform(scaler.transform(X_cal)))
    test_pred = model.predict(selector.transform(scaler.transform(X_test)))
    residuals = calibration[target].to_numpy(float) - cal_pred
    p10, p90 = np.percentile(residuals, [10, 90])
    out = test_df[['date', vessel, target]].copy()
    out = out.rename(columns={vessel: 'current_rate', target: 'actual_rate'})
    out['vessel'] = vessel
    out['prediction'] = test_pred
    out['lower_bound'] = np.maximum(0.0, test_pred + p10)
    out['median'] = test_pred
    out['upper_bound'] = np.maximum(0.0, test_pred + p90)
    out['calibration_p10_residual'] = p10
    out['calibration_p90_residual'] = p90
    out['train_end'] = core.date.max().strftime('%Y-%m-%d')
    out['test_year'] = int(test_df.date.dt.year.iloc[0])
    out['model_id'] = 'RF_STANDARD_FRESH_WALK_FORWARD'
    return out

predictions = []
for year in OOS_YEARS:
    train = df[df.date.dt.year < year]
    test = df[df.date.dt.year == year]
    for vessel in VESSELS:
        predictions.append(fit_fold_predict(train, test, vessel))
predictions = pd.concat(predictions, ignore_index=True)
predictions['error'] = predictions['prediction'] - predictions['actual_rate']
predictions['direction_actual'] = np.sign(predictions['actual_rate'] - predictions['current_rate'])
predictions['direction_predicted'] = np.sign(predictions['prediction'] - predictions['current_rate'])
predictions.to_csv(OUT / 'oos_predictions.csv', index=False)
print('Fresh OOS predictions:', len(predictions))
print(predictions[['vessel', 'test_year', 'train_end']].drop_duplicates().to_string(index=False))

In [ ]:
# Raw forecast metrics, calculated from the new prediction table in memory.
forecast_metrics = []
for (vessel, year), group in predictions.groupby(['vessel', 'test_year']):
    forecast_metrics.append({
        'vessel': vessel, 'test_year': int(year), 'n': len(group),
        'mae': float(np.abs(group.error).mean()),
        'rmse': float(np.sqrt(np.mean(group.error ** 2))),
        'directional_accuracy': float((group.direction_actual == group.direction_predicted).mean()),
        'interval_coverage': float(((group.actual_rate >= group.lower_bound) & (group.actual_rate <= group.upper_bound)).mean()),
        'mean_interval_width': float((group.upper_bound - group.lower_bound).mean()),
    })
forecast_metrics = pd.DataFrame(forecast_metrics)
forecast_metrics.to_csv(OUT / 'forecast_metrics.csv', index=False)
print(forecast_metrics.to_string(index=False))

## 2. Quantify timing uncertainty and fresh OOS economic value

This is a transparent timing proxy computed from the fresh OOS rows:

- baseline: buy at the decision-time current rate;
- policy: buy now when the forecast rises beyond 1%, wait when it falls beyond 1%, otherwise buy now;
- realized policy cost: current rate for buy-now, next-day realized rate for wait;
- savings: baseline cost minus realized policy cost.

This is **not** the frozen EXP-04/EXP-06 economic result and does not reuse its outputs.

In [ ]:
economic = predictions.copy()
economic['forecast_delta'] = economic.prediction - economic.current_rate
economic['threshold'] = 0.01 * economic.current_rate
economic['timing_decision'] = np.where(economic.forecast_delta < -economic.threshold, 'WAIT', 'BUY_NOW')
economic['baseline_cost_per_mt'] = economic.current_rate
economic['policy_cost_per_mt'] = np.where(economic.timing_decision == 'WAIT', economic.actual_rate, economic.current_rate)
economic['baseline_cost_usd'] = economic.baseline_cost_per_mt * 75_000
economic['policy_cost_usd'] = economic.policy_cost_per_mt * 75_000
economic['savings_usd'] = economic.baseline_cost_usd - economic.policy_cost_usd
economic['data_status'] = 'LOCKED_OOS_FROM_FRESH_WALK_FORWARD'
economic.to_csv(OUT / 'fresh_oos_economic_results.csv', index=False)
economic_summary = economic.groupby('vessel').agg(n=('vessel','size'), wait_decisions=('timing_decision', lambda x: int((x == 'WAIT').sum())), total_savings_usd=('savings_usd','sum'), mean_savings_usd=('savings_usd','mean')).reset_index()
economic_summary.loc[len(economic_summary)] = ['ALL', len(economic), economic.timing_decision.eq('WAIT').sum(), economic.savings_usd.sum(), economic.savings_usd.mean()]
economic_summary.to_csv(OUT / 'fresh_oos_economic_summary.csv', index=False)
print(economic_summary.to_string(index=False))

## 3. Represent voyages and expose cross-voyage coupling

The last 12 fresh OOS forecasts become a small research voyage set. Rate and forecast fields are data-derived. Volume, duration, capacity, contract discounts, budget, and contract capacity are scenario assumptions because they are absent from the canonical dataset.

In [ ]:
from ml.planning import Voyage, VoyageOpportunity, PlanningConstraints, DeterministicMILPPlanner, RiskAwareMILPPlanner, Strategy, generate_rate_scenarios

# Keep the optimization fixture small and auditable while preserving fresh OOS provenance.
selected_rows = predictions.sort_values(['date', 'vessel']).tail(12).reset_index(drop=True)
opportunities = []
for i, row in selected_rows.iterrows():
    voyage = Voyage(f'FRESH-OOS-{i+1:02d}', row.vessel.upper(), 'SCENARIO_ROUTE', 'SCENARIO_ORIGIN', 'SCENARIO_DESTINATION', row.date.strftime('%Y-%m-%d'), row.date.strftime('%Y-%m-%d'), 20, 75_000.0, 82_000.0, row.date.strftime('%Y-%m-%d'), {'data_status': 'SCENARIO_ASSUMPTION_FOR_MISSING_VOYAGE_FIELDS', 'source_oos_date': row.date.strftime('%Y-%m-%d')})
    base = max(1.0, float(row.prediction)) * voyage.volume_mt * voyage.expected_duration_days
    costs = {'SPOT': base, 'SHORT_TERM': base * .985, 'MEDIUM_TERM': base * .970, 'MULTI_VOYAGE_CONTRACT': base * (.955 if i < 6 else .965)}
    opportunities.append(VoyageOpportunity(voyage, float(row.current_rate), float(row.prediction), float(row.lower_bound), float(row.upper_bound), costs, 'RISING' if row['prediction'] > row['current_rate'] else 'FALLING', 'calibration residual P10/P90', ('SCENARIO_ASSUMPTION: volume', 'SCENARIO_ASSUMPTION: duration', 'SCENARIO_ASSUMPTION: contract discount')))
opportunity_table = pd.DataFrame([o.to_dict() for o in opportunities])
opportunity_table.to_csv(OUT / 'fresh_voyage_opportunities.csv', index=False)
print(opportunity_table[['voyage_id','vessel_class','forecast_rate','lower_rate','upper_rate','signal']].to_string(index=False))

## 4. Optimize contracts and timing

The planner uses a real MILP. The same fresh opportunities are evaluated under deterministic, robust, and mean-CVaR objectives. Coupling is represented through shared contract capacity, budget, and one strategy per voyage.

In [ ]:
constraints = PlanningConstraints(budget_usd=sum(o.strategy_costs['SPOT'] for o in opportunities) * 1.02, contract_capacity_mt=400_000, max_contracts=len(opportunities))
deterministic = DeterministicMILPPlanner().solve(opportunities, constraints)
residuals = predictions['error'].to_numpy(float)
rate_scenarios = generate_rate_scenarios([o.forecast_rate for o in opportunities], residuals, n_scenarios=100, seed=SEED)
scenario_costs = np.zeros((len(rate_scenarios), len(opportunities), len(Strategy)))
for s, scenario_rates in enumerate(rate_scenarios):
    for i, opportunity in enumerate(opportunities):
        base = scenario_rates[i] * opportunity.voyage.volume_mt * opportunity.voyage.expected_duration_days
        scenario_costs[s, i] = [base, base*.985, base*.970, base*(.955 if i < 6 else .965)]
planner = RiskAwareMILPPlanner()
robust = planner.solve_risk_aware(opportunities, scenario_costs, constraints, mode='ROBUST')
# Scale both objective coefficients and opportunity costs for CVaR numerical conditioning.
scaled_opportunities = [VoyageOpportunity(o.voyage, o.current_rate, o.forecast_rate, o.lower_rate, o.upper_rate, {k: v / 1_000_000 for k, v in o.strategy_costs.items()}, o.signal, o.uncertainty_method, o.assumptions) for o in opportunities]
scaled_constraints = PlanningConstraints(budget_usd=constraints.budget_usd / 1_000_000, contract_capacity_mt=constraints.contract_capacity_mt, max_contracts=constraints.max_contracts)
mean_cvar = planner.solve_risk_aware(scaled_opportunities, scenario_costs / 1_000_000, scaled_constraints, mode='MEAN_CVAR', alpha=.90, risk_aversion=.25)
pd.DataFrame(rate_scenarios).to_csv(OUT / 'fresh_rate_scenarios.csv', index=False)
optimization_result = {'deterministic': deterministic.to_dict(), 'robust': robust.to_dict(), 'mean_cvar': mean_cvar.to_dict(), 'cvar_objective_units': 'million_USD', 'constraints': {'budget_usd': constraints.budget_usd, 'contract_capacity_mt': constraints.contract_capacity_mt, 'max_contracts': constraints.max_contracts, 'all_non_observed_fields': 'SCENARIO_ASSUMPTION'}}
(OUT / 'fresh_optimization_result.json').write_text(json.dumps(optimization_result, indent=2), encoding='utf-8')
print(json.dumps(optimization_result, indent=2))

## 5. Final raw evidence

The summary below is generated by this notebook run. It is not copied from repository reports.

In [ ]:
summary = {
    'experiment_id': 'FICOS_FRESH_PIPELINE_SYSTEM_TEST',
    'dataset_sha256': sha256(DATA_PATH),
    'rows': int(len(df)), 'features': int(len(feature_cols)),
    'fresh_oos_predictions': int(len(predictions)),
    'forecast_models': 'RF_STANDARD, refit inside each expanding fold',
    'uncertainty': 'calibration-tail empirical P10/P90 residual bounds',
    'voyage_opportunities': int(len(opportunities)),
    'deterministic_milp_status': deterministic.solver_status,
    'robust_milp_status': robust.solver_status,
    'mean_cvar_milp_status': mean_cvar.solver_status,
    'fresh_oos_savings_usd': float(economic.savings_usd.sum()),
    'fresh_oos_wait_decisions': int(economic.timing_decision.eq('WAIT').sum()),
    'interpretation': 'Fresh OOS timing proxy under stated formula; not EXP-04/EXP-06 and not a claim of causal commercial savings.',
}
(OUT / 'fresh_pipeline_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('RAW FILES:', sorted(p.name for p in OUT.iterdir()))